# Mixed-Frequency GDP Nowcasting — Tracking the Real-Time Economy

**How can central banks and fiscal authorities estimate current-quarter GDP growth in real time before official statistical releases?**

Macroeconomic indicators arrive at high frequencies (monthly/weekly) and with staggered publication delays, creating **ragged edges** at the boundary of the sample.

Domenico Giannone, Lucrezia Reichlin, and David Small (2008, *Journal of Monetary Economics*) introduced the state-space **Dynamic Factor Model (DFM)** with missing data. The framework accomplishes two critical goals:
1. **Real-Time Nowcasting**: Bridges monthly informational flows to quarterly GDP via unobserved common factors $F_t$.
2. **News Decomposition**: Quantifies the surprise component in every fresh data release and its precise impact on the nowcast revision.

In this interactive showcase, we track GDP growth and decompose data releases using `puremacro.nowcast.nowcast_gdp`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_cwd = Path.cwd()
sys.path.insert(0, str(_cwd if (_cwd / "_nbstyle.py").exists() else _cwd / "notebooks"))
import _nbstyle
_nbstyle.apply_style()

from puremacro.nowcast import nowcast_gdp

## 1. Simulating an Asynchronous Macroeconomic Panel

We observe 10 monthly indicators across real activity, labor, inflation, and sentiment. The latest month contains missing values (ragged edge) representing publication lags.

In [ ]:
rng = np.random.default_rng(42)
n_months = 72
dates_m = pd.date_range("2018-01-01", periods=n_months, freq="MS")

F_true = np.zeros((n_months, 2))
for t in range(1, n_months):
    F_true[t, 0] = 0.85 * F_true[t-1, 0] + rng.normal(scale=0.8)
    F_true[t, 1] = 0.60 * F_true[t-1, 1] + rng.normal(scale=0.5)

var_names = [
    "Industrial Production", "Payroll Employment", "Retail Sales",
    "Housing Starts", "Capacity Utilization", "PMI Manufacturing",
    "Core CPI Inflation", "Real Personal Income", "Export Orders", "Consumer Sentiment"
]
N = len(var_names)
X = np.zeros((n_months, N))
for i in range(N):
    load = rng.uniform(0.4, 1.6, size=2)
    X[:, i] = F_true @ load + rng.normal(scale=0.4, size=n_months)

df_X = pd.DataFrame(X, index=dates_m, columns=var_names)
# Staggered release delays
df_X.iloc[-1, [3, 4, 8, 9]] = np.nan

# Historical quarterly GDP
dates_q = pd.date_range("2018-01-01", periods=n_months // 3, freq="QS")
F_q = df_X.resample("QE").mean().to_numpy().mean(axis=1)[:len(dates_q)]
gdp = 2.2 + 1.4 * F_q + rng.normal(scale=0.3, size=len(dates_q))
s_gdp = pd.Series(gdp, index=dates_q.to_period("Q").astype(str), name="GDP Growth")

## 2. Estimating the Dynamic Factor Nowcast & News Decomposition

In [ ]:
res = nowcast_gdp(df_X, s_gdp, n_factors=2)
print(res.summary())

## 3. Visualizing Latent Monthly Factors & Factor Loadings

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(df_X.index, res.factors["Factor_1"], color="#1f77b4", lw=2, label="Factor 1 (Real Activity)")
ax1.plot(df_X.index, res.factors["Factor_2"], color="#ff7f0e", lw=2, linestyle="--", label="Factor 2 (Demand/Sentiment)")
ax1.set_title("Smoothed Monthly Common Factors", fontsize=11, fontweight="bold")
ax1.set_xlabel("Date")
ax1.set_ylabel("Factor Level")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

im = ax2.imshow(res.loadings.to_numpy(), cmap="coolwarm", aspect="auto")
ax2.set_yticks(range(N))
ax2.set_yticklabels(var_names, fontsize=8)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(["Factor 1", "Factor 2"])
ax2.set_title("Estimated Factor Loadings Λ", fontsize=11, fontweight="bold")
fig.colorbar(im, ax=ax2, label="Weight")

plt.tight_layout()
plt.show()

## 4. News and Revision Decomposition

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

if not res.news_decomposition.empty:
    bars1 = ax1.barh(res.news_decomposition["series"], res.news_decomposition["surprise"], color="#2ca02c", edgecolor="#333")
    for b in bars1:
        if b.get_width() < 0:
            b.set_color("#d62728")
    ax1.set_title("Data Release Surprises (Actual - Forecast)", fontsize=11, fontweight="bold")
    ax1.set_xlabel("Surprise (σ units)")
    ax1.grid(True, linestyle=":", alpha=0.6)

    bars2 = ax2.barh(res.news_decomposition["series"], res.news_decomposition["contribution"], color="#1f77b4", edgecolor="#333")
    for b in bars2:
        if b.get_width() < 0:
            b.set_color("#d62728")
    ax2.set_title("Impact on GDP Nowcast Revision", fontsize=11, fontweight="bold")
    ax2.set_xlabel("Contribution (percentage points)")
    ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()